In [1]:
# =========================
# CELL 1: compute_dynamic_features (corrected)
# =========================

from pathlib import Path
import pandas as pd
import numpy as np
from scipy.signal import savgol_filter

def find_root():
    p = Path.cwd()
    while p != p.parent:
        if (p / "data").exists() and (p / "scripts").exists():
            return p
        p = p.parent
    raise RuntimeError("Root folder not found")

ROOT = find_root()

INPUT_DIR = ROOT / "data" / "processed" / "aggregated_by_signal"
OUTPUT_FILE = ROOT / "data" / "processed" / "features" / "analysis_dataset_dynamic.csv"

BIN_SECONDS = 10

print("INPUT_DIR:", INPUT_DIR)
print("OUTPUT_FILE:", OUTPUT_FILE)


def assign_phase(elapsed_min):
    if elapsed_min < 10:
        return "standing_1"
    elif elapsed_min < 20:
        return "walk_low_1"
    elif elapsed_min < 30:
        return "walk_mod_1"
    elif elapsed_min < 40:
        return "walk_low_2"
    elif elapsed_min < 50:
        return "walk_mod_2"
    else:
        return "standing_2"


def load_signal(signal_file: str, value_name: str) -> pd.DataFrame:
    path = INPUT_DIR / signal_file
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")

    df = pd.read_csv(path)

    keep_cols = ["subject_id", "condition", "elapsed_seconds", "elapsed_minutes", "value"]
    df = df[keep_cols].copy()

    df["subject_id"] = df["subject_id"].astype(str).str.strip()
    df["condition"] = df["condition"].astype(str).str.strip()
    df["elapsed_seconds"] = pd.to_numeric(df["elapsed_seconds"], errors="coerce")
    df["elapsed_minutes"] = pd.to_numeric(df["elapsed_minutes"], errors="coerce")
    df["value"] = pd.to_numeric(df["value"], errors="coerce")

    df = df.dropna(subset=["subject_id", "condition", "elapsed_seconds", "value"]).copy()

    # temporal binning
    df["elapsed_seconds_bin"] = (df["elapsed_seconds"] / BIN_SECONDS).round() * BIN_SECONDS
    df["elapsed_minutes_bin"] = df["elapsed_seconds_bin"] / 60.0

    df = (
        df.groupby(
            ["subject_id", "condition", "elapsed_seconds_bin", "elapsed_minutes_bin"],
            as_index=False
        )["value"]
        .mean()
    )

    df = df.rename(columns={"value": value_name})
    return df


# -------------------------
# Load and merge core signals
# -------------------------
dorsal = load_signal("t_dorsal.csv", "t_dorsal")
micro = load_signal("t_microclimateback.csv", "t_microclimateback")

df = pd.merge(
    dorsal,
    micro,
    on=["subject_id", "condition", "elapsed_seconds_bin", "elapsed_minutes_bin"],
    how="inner"
)

df["phase"] = df["elapsed_minutes_bin"].apply(assign_phase)
df["delta_t_loc"] = df["t_dorsal"] - df["t_microclimateback"]

print("Merged dynamic dataframe shape:", df.shape)
print(df["condition"].value_counts(dropna=False))
print(df.groupby("condition")["delta_t_loc"].apply(lambda x: x.notna().sum()))

# -------------------------
# Smoothing + derivative per subject/condition
# -------------------------
def compute_dynamic_group(g):
    g = g.sort_values("elapsed_seconds_bin").copy()

    y = g["t_dorsal"].to_numpy(dtype=float)
    t = g["elapsed_seconds_bin"].to_numpy(dtype=float)

    mask = np.isfinite(y) & np.isfinite(t)

    g["t_dorsal_smooth"] = np.nan
    g["dt_dorsal_dt"] = np.nan
    g["minus_dt_dorsal_dt"] = np.nan

    if mask.sum() < 15:
        return g

    y_valid = y[mask]
    t_valid = t[mask]

    window = 11 if len(y_valid) >= 11 else len(y_valid) // 2 * 2 + 1
    if window < 5:
        return g

    y_smooth = savgol_filter(y_valid, window_length=window, polyorder=2)

    dt = np.gradient(t_valid)
    dy = np.gradient(y_smooth)
    derivative = dy / dt

    g.loc[mask, "t_dorsal_smooth"] = y_smooth
    g.loc[mask, "dt_dorsal_dt"] = derivative
    g.loc[mask, "minus_dt_dorsal_dt"] = -derivative

    return g


df = df.groupby(["subject_id", "condition"], group_keys=False).apply(compute_dynamic_group)

# -------------------------
# HTI_app
# -------------------------
df["hti_app"] = df["delta_t_loc"] / df["minus_dt_dorsal_dt"]
df.replace([np.inf, -np.inf], np.nan, inplace=True)

# -------------------------
# Save
# -------------------------
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUTPUT_FILE, index=False)

print("Saved:", OUTPUT_FILE)
print("Non-null by condition:")
print(df.groupby("condition")["delta_t_loc"].apply(lambda x: x.notna().sum()))
print(df.groupby("condition")["minus_dt_dorsal_dt"].apply(lambda x: x.notna().sum()))
print(df.groupby("condition")["hti_app"].apply(lambda x: x.notna().sum()))

INPUT_DIR: /home/eleonorab/repository/convective_cooling/data/processed/aggregated_by_signal
OUTPUT_FILE: /home/eleonorab/repository/convective_cooling/data/processed/features/analysis_dataset_dynamic.csv
Merged dynamic dataframe shape: (4674, 8)
condition
wind_high    1796
no_wind      1440
wind_low     1438
Name: count, dtype: int64
condition
no_wind      1440
wind_high    1796
wind_low     1438
Name: delta_t_loc, dtype: int64
Saved: /home/eleonorab/repository/convective_cooling/data/processed/features/analysis_dataset_dynamic.csv
Non-null by condition:
condition
no_wind      1440
wind_high    1796
wind_low     1438
Name: delta_t_loc, dtype: int64
condition
no_wind      1440
wind_high    1796
wind_low     1438
Name: minus_dt_dorsal_dt, dtype: int64
condition
no_wind      1219
wind_high    1625
wind_low     1293
Name: hti_app, dtype: int64


/tmp/ipykernel_13026/379614105.py:136: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby(["subject_id", "condition"], group_keys=False).apply(compute_dynamic_group)


In [2]:
# =========================
# CELL 2: extract_dynamic_phase_features (corrected)
# =========================

import pandas as pd
import numpy as np
from pathlib import Path

ROOT = find_root()

INPUT_FILE = ROOT / "data" / "processed" / "features" / "analysis_dataset_dynamic.csv"
OUTPUT_FILE = ROOT / "data" / "processed" / "features" / "dynamic_phase_features.csv"

df = pd.read_csv(INPUT_FILE)

PHASE_ORDER = [
    "standing_1",
    "walk_low_1",
    "walk_mod_1",
    "walk_low_2",
    "walk_mod_2",
    "standing_2",
]

signals = ["delta_t_loc", "hti_app", "minus_dt_dorsal_dt"]

# pulizia tipi
df["subject_id"] = df["subject_id"].astype(str).str.strip()
df["condition"] = df["condition"].astype(str).str.strip()
df["phase"] = df["phase"].astype(str).str.strip()

for col in ["elapsed_minutes_bin", "elapsed_seconds_bin", *signals]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

df.replace([np.inf, -np.inf], np.nan, inplace=True)

def compute_auc(x, y):
    if len(x) < 2:
        return np.nan
    return np.trapz(y, x)

rows = []

for (subj, cond, phase), g in df.groupby(["subject_id", "condition", "phase"], observed=True):
    g = g.sort_values("elapsed_minutes_bin").copy()

    for sig in signals:
        if sig not in g.columns:
            continue

        y = pd.to_numeric(g[sig], errors="coerce")
        x = pd.to_numeric(g["elapsed_minutes_bin"], errors="coerce")

        valid = pd.DataFrame({"x": x, "y": y}).dropna()

        if valid.empty:
            continue

        last_2min = valid[valid["x"] >= (valid["x"].max() - 2)]

        rows.append({
            "subject_id": subj,
            "condition": cond,
            "phase": phase,
            "signal": sig,
            "n_samples": len(valid),
            "mean": valid["y"].mean(),
            "median": valid["y"].median(),
            "p25": valid["y"].quantile(0.25),
            "p75": valid["y"].quantile(0.75),
            "mean_last2min": last_2min["y"].mean() if not last_2min.empty else np.nan,
            "auc": compute_auc(valid["x"].to_numpy(dtype=float), valid["y"].to_numpy(dtype=float)),
        })

df_feat = pd.DataFrame(rows)

if not df_feat.empty:
    df_feat["phase"] = pd.Categorical(df_feat["phase"], categories=PHASE_ORDER, ordered=True)
    df_feat = df_feat.sort_values(["signal", "condition", "subject_id", "phase"]).reset_index(drop=True)

OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
df_feat.to_csv(OUTPUT_FILE, index=False)

print("Saved:", OUTPUT_FILE)
print("Shape:", df_feat.shape)
print(df_feat.head())

print("\nRows by signal/condition:")
if not df_feat.empty:
    print(df_feat.groupby(["signal", "condition"]).size())

Saved: /home/eleonorab/repository/convective_cooling/data/processed/features/dynamic_phase_features.csv
Shape: (234, 11)
  subject_id condition       phase       signal  n_samples      mean  median  \
0  tester_01   no_wind  standing_1  delta_t_loc         59 -1.109424 -1.2030   
1  tester_01   no_wind  walk_low_1  delta_t_loc         60 -0.299100 -0.4235   
2  tester_01   no_wind  walk_mod_1  delta_t_loc         60  0.221533  0.1690   
3  tester_01   no_wind  walk_low_2  delta_t_loc         60  0.286250  0.2940   
4  tester_01   no_wind  walk_mod_2  delta_t_loc         60  0.482600  0.4800   

       p25     p75  mean_last2min        auc  
0 -1.39000 -0.7980      -1.016250 -10.771167  
1 -0.64225  0.1215       0.341692  -2.962000  
2  0.16800  0.2940       0.158769   2.176833  
3  0.16900  0.3560       0.317462   2.818833  
4  0.41800  0.5430       0.581538   4.735500  

Rows by signal/condition:
signal              condition
delta_t_loc         no_wind      24
                    win

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = find_root()
INPUT_FILE = ROOT / "data" / "processed" / "features" / "dynamic_phase_features.csv"
OUTPUT_DIR = ROOT / "reports" / "dynamic_feature_plots"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PHASE_ORDER = [
    "standing_1",
    "walk_low_1",
    "walk_mod_1",
    "walk_low_2",
    "walk_mod_2",
    "standing_2",
]
CONDITION_ORDER = ["no_wind", "wind_low", "wind_high"]

df_feat = pd.read_csv(INPUT_FILE)
df_feat.replace([np.inf, -np.inf], np.nan, inplace=True)

df_feat["condition"] = df_feat["condition"].astype(str).str.strip()
df_feat["phase"] = df_feat["phase"].astype(str).str.strip()
df_feat["signal"] = df_feat["signal"].astype(str).str.strip()

for col in ["mean", "median", "p25", "p75", "mean_last2min", "auc"]:
    if col in df_feat.columns:
        df_feat[col] = pd.to_numeric(df_feat[col], errors="coerce")


def plot_phase_feature(df, signal, metric, filename):
    sub = df[df["signal"] == signal].copy()
    sub = sub[np.isfinite(sub[metric])].copy()

    print(f"\n{signal} | {metric} | rows:", len(sub))
    print(sub.groupby(["condition", "phase"])[metric].count())

    if sub.empty:
        print(f"Skipping {signal} - {metric}: no valid data")
        return

    plot_df = (
        sub.groupby(["condition", "phase"], observed=True)[metric]
        .agg(["mean", "std", "count"])
        .reset_index()
    )

    plot_df["sem"] = plot_df["std"] / np.sqrt(plot_df["count"])
    phase_to_x = {phase: i for i, phase in enumerate(PHASE_ORDER)}

    plt.figure(figsize=(8, 5))
    plotted = 0

    for condition in CONDITION_ORDER:
        cond_df = plot_df[plot_df["condition"] == condition].copy()
        if cond_df.empty:
            continue

        cond_df["phase"] = pd.Categorical(cond_df["phase"], categories=PHASE_ORDER, ordered=True)
        cond_df = cond_df.sort_values("phase")

        x = [phase_to_x[p] for p in cond_df["phase"].astype(str)]
        y = cond_df["mean"].to_numpy(dtype=float)
        sem = cond_df["sem"].fillna(0).to_numpy(dtype=float)

        plt.plot(x, y, marker="o", label=condition)
        plt.fill_between(x, y - sem, y + sem, alpha=0.2)
        plotted += 1

    if plotted == 0:
        print(f"Skipping {signal} - {metric}: nothing plotted")
        plt.close()
        return

    plt.xticks(list(phase_to_x.values()), list(phase_to_x.keys()), rotation=45)
    plt.xlabel("Phase", fontsize=12)
    plt.ylabel(f"{signal} - {metric}", fontsize=12)
    plt.title(f"{signal}: {metric} by phase and condition", fontsize=14)
    plt.legend()
    plt.tight_layout()

    out_path = OUTPUT_DIR / filename
    plt.savefig(out_path, dpi=300)
    plt.close()
    print("Saved:", out_path)


plot_phase_feature(df_feat, "delta_t_loc", "mean", "delta_t_loc_mean_by_phase.png")
plot_phase_feature(df_feat, "minus_dt_dorsal_dt", "mean", "minus_dt_dorsal_dt_mean_by_phase.png")
plot_phase_feature(df_feat, "hti_app", "mean", "hti_app_mean_by_phase.png")
plot_phase_feature(df_feat, "hti_app", "mean_last2min", "hti_app_mean_last2min_by_phase.png")
plot_phase_feature(df_feat, "delta_t_loc", "auc", "delta_t_loc_auc_by_phase.png")


delta_t_loc | mean | rows: 78
condition  phase     
no_wind    standing_1    4
           standing_2    4
           walk_low_1    4
           walk_low_2    4
           walk_mod_1    4
           walk_mod_2    4
wind_high  standing_1    5
           standing_2    5
           walk_low_1    5
           walk_low_2    5
           walk_mod_1    5
           walk_mod_2    5
wind_low   standing_1    4
           standing_2    4
           walk_low_1    4
           walk_low_2    4
           walk_mod_1    4
           walk_mod_2    4
Name: mean, dtype: int64
Saved: /home/eleonorab/repository/convective_cooling/reports/dynamic_feature_plots/delta_t_loc_mean_by_phase.png

minus_dt_dorsal_dt | mean | rows: 78
condition  phase     
no_wind    standing_1    4
           standing_2    4
           walk_low_1    4
           walk_low_2    4
           walk_mod_1    4
           walk_mod_2    4
wind_high  standing_1    5
           standing_2    5
           walk_low_1    5
           walk_low_

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pathlib import Path

ROOT = find_root()
INPUT_FILE = ROOT / "data" / "processed" / "features" / "dynamic_phase_features.csv"
OUTPUT_DIR = ROOT / "reports" / "dynamic_feature_plots"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df_feat = pd.read_csv(INPUT_FILE)
df_feat.replace([np.inf, -np.inf], np.nan, inplace=True)

df_feat["condition"] = df_feat["condition"].astype(str).str.strip()
df_feat["phase"] = df_feat["phase"].astype(str).str.strip()
df_feat["signal"] = df_feat["signal"].astype(str).str.strip()

for col in ["mean", "mean_last2min", "auc"]:
    if col in df_feat.columns:
        df_feat[col] = pd.to_numeric(df_feat[col], errors="coerce")

PHASES_OF_INTEREST = ["walk_mod_1", "walk_mod_2", "standing_2"]
CONDITION_ORDER = ["no_wind", "wind_low", "wind_high"]


def plot_condition_distribution(df, signal, metric, phases, prefix):
    sub = df[
        (df["signal"] == signal) &
        (df["phase"].isin(phases))
    ].copy()

    for phase in phases:
        phase_df = sub[sub["phase"] == phase].copy()
        phase_df = phase_df[np.isfinite(phase_df[metric])].copy()

        print(f"\n{signal} | {metric} | {phase}")
        if phase_df.empty:
            print("No valid data")
            continue

        print(phase_df.groupby("condition")[metric].count())

        plt.figure(figsize=(7, 5))

        sns.violinplot(
            data=phase_df,
            x="condition",
            y=metric,
            order=CONDITION_ORDER,
            inner=None,
            cut=0
        )

        sns.stripplot(
            data=phase_df,
            x="condition",
            y=metric,
            order=CONDITION_ORDER,
            dodge=False,
            alpha=0.8
        )

        plt.xlabel("Condition", fontsize=12)
        plt.ylabel(f"{signal} - {metric}", fontsize=12)
        plt.title(f"{signal}: {metric} by condition - {phase}", fontsize=14)
        plt.tight_layout()

        out_path = OUTPUT_DIR / f"{prefix}_{signal}_{metric}_{phase}.png"
        plt.savefig(out_path, dpi=300)
        plt.close()

        print("Saved:", out_path)


plot_condition_distribution(df_feat, "delta_t_loc", "mean", PHASES_OF_INTEREST, "condition_distribution")
plot_condition_distribution(df_feat, "minus_dt_dorsal_dt", "mean", PHASES_OF_INTEREST, "condition_distribution")
plot_condition_distribution(df_feat, "hti_app", "mean", PHASES_OF_INTEREST, "condition_distribution")


delta_t_loc | mean | walk_mod_1
condition
no_wind      4
wind_high    5
wind_low     4
Name: mean, dtype: int64
Saved: /home/eleonorab/repository/convective_cooling/reports/dynamic_feature_plots/condition_distribution_delta_t_loc_mean_walk_mod_1.png

delta_t_loc | mean | walk_mod_2
condition
no_wind      4
wind_high    5
wind_low     4
Name: mean, dtype: int64
Saved: /home/eleonorab/repository/convective_cooling/reports/dynamic_feature_plots/condition_distribution_delta_t_loc_mean_walk_mod_2.png

delta_t_loc | mean | standing_2
condition
no_wind      4
wind_high    5
wind_low     4
Name: mean, dtype: int64
Saved: /home/eleonorab/repository/convective_cooling/reports/dynamic_feature_plots/condition_distribution_delta_t_loc_mean_standing_2.png

minus_dt_dorsal_dt | mean | walk_mod_1
condition
no_wind      4
wind_high    5
wind_low     4
Name: mean, dtype: int64
Saved: /home/eleonorab/repository/convective_cooling/reports/dynamic_feature_plots/condition_distribution_minus_dt_dorsal_dt_me

In [12]:
# ============================================================
# Dynamic descriptors figure
# Uses: convective_indices_subject_phase_summary.csv
# ============================================================

# ------------------------------------------------------------
# Global font settings
# ------------------------------------------------------------

plt.rcParams.update({
    "font.size": 16,          # base font
    "axes.titlesize": 18,     # subplot titles
    "axes.labelsize": 17,     # axis labels
    "xtick.labelsize": 15,    # x ticks
    "ytick.labelsize": 15,    # y ticks
    "legend.fontsize": 15,    # legend
    "figure.titlesize": 20,   # main figure title
})

def find_root() -> Path:
    p = Path.cwd()
    while p != p.parent:
        if (p / "data").exists() and (p / "scripts").exists():
            return p
        p = p.parent
    raise RuntimeError("Root folder not found")


ROOT = find_root()

INPUT_FILE = (
    ROOT
    / "reports"
    / "mechanistic_plots"
    / "convective_indices_example"
    / "convective_indices_subject_phase_summary.csv"
)

# fallback if running with uploaded file in ChatGPT sandbox
if not INPUT_FILE.exists():
    INPUT_FILE = Path("/mnt/data/convective_indices_subject_phase_summary.csv")

OUTPUT_DIR = ROOT / "reports" / "mechanistic_plots" / "convective_indices_example"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# Settings
# ------------------------------------------------------------

CONDITION_ORDER = ["no_wind", "wind_low", "wind_high"]

CONDITION_LABELS = {
    "no_wind": "No ventilation",
    "wind_low": "Low ventilation",
    "wind_high": "High ventilation",
}

CONDITION_COLORS = {
    "no_wind": "blue",
    "wind_low": "red",
    "wind_high": "black",
}

PHASE_ORDER = [
    "standing_1",
    "walk_low_1",
    "walk_mod_1",
    "walk_low_2",
    "walk_mod_2",
    "standing_2",
]

PHASE_LABELS = {
    "standing_1": "Phase 1",
    "walk_low_1": "Phase 2",
    "walk_mod_1": "Phase 3",
    "walk_low_2": "Phase 4",
    "walk_mod_2": "Phase 5",
    "standing_2": "Phase 6",
}

# ------------------------------------------------------------
# Load and clean
# ------------------------------------------------------------

df = pd.read_csv(INPUT_FILE)

numeric_cols = [
    "mean_deltaT_loc",
    "mean_CR_C_per_s",
    "mean_ACI_1_per_s",
    "k_c_slope_1_per_s",
    "k_transition_1_per_s",
    "tau_transition_s",
    "r_squared",
    "r_squared_transition",
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

df["condition"] = pd.Categorical(
    df["condition"].astype(str).str.strip(),
    categories=CONDITION_ORDER,
    ordered=True,
)

df["phase"] = pd.Categorical(
    df["phase"].astype(str).str.strip(),
    categories=PHASE_ORDER,
    ordered=True,
)

df = df.dropna(subset=["condition", "phase"]).copy()


# ------------------------------------------------------------
# Derived descriptor: CCP
# ------------------------------------------------------------
# CCP is the cumulative integral of DeltaT_loc over each phase.
# Since each phase is 10 min = 600 s, this approximation is:
# CCP = mean_deltaT_loc * 600
# Unit: °C s

PHASE_DURATION_S = 600

df["CCP_deltaT_C_s"] = df["mean_deltaT_loc"] * PHASE_DURATION_S


# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------

def add_boxplot_with_points(ax, data, variable, ylabel, title, log_scale=False):
    plot_df = data.dropna(subset=[variable]).copy()

    values = [
        plot_df.loc[plot_df["condition"] == cond, variable].values
        for cond in CONDITION_ORDER
    ]

    bp = ax.boxplot(
    values,
    labels=[CONDITION_LABELS[c] for c in CONDITION_ORDER],
    showmeans=True,
    widths=0.55,
    patch_artist=True,
    )

    for patch, cond in zip(bp["boxes"], CONDITION_ORDER):
        patch.set_facecolor("white")
        patch.set_edgecolor(CONDITION_COLORS[cond])
        patch.set_linewidth(1.8)

    for whisker, cond in zip(bp["whiskers"], np.repeat(CONDITION_ORDER, 2)):
        whisker.set_color(CONDITION_COLORS[cond])
        whisker.set_linewidth(1.5)

    for cap, cond in zip(bp["caps"], np.repeat(CONDITION_ORDER, 2)):
        cap.set_color(CONDITION_COLORS[cond])
        cap.set_linewidth(1.5)

    for median, cond in zip(bp["medians"], CONDITION_ORDER):
        median.set_color(CONDITION_COLORS[cond])
        median.set_linewidth(2)

    for mean, cond in zip(bp["means"], CONDITION_ORDER):
        mean.set_markerfacecolor(CONDITION_COLORS[cond])
        mean.set_markeredgecolor(CONDITION_COLORS[cond])

    rng = np.random.default_rng(42)

    for i, cond in enumerate(CONDITION_ORDER, start=1):
        vals = plot_df.loc[plot_df["condition"] == cond, variable].dropna().values
        jitter = rng.normal(0, 0.04, size=len(vals))

    ax.scatter(
        np.full(len(vals), i) + jitter,
        vals,
        alpha=0.45,
        s=24,
        color=CONDITION_COLORS[cond],
    )

    if log_scale:
        ax.set_yscale("log")

    ax.set_ylabel(ylabel, fontsize=14)
    ax.set_title(title, fontsize=16)
    ax.tick_params(axis="x", rotation=25, labelsize=15)
    ax.tick_params(axis="y", labelsize=15)
    ax.spines[["top", "right"]].set_visible(False)


def add_phase_lineplot(ax, data, variable, ylabel, title):
    plot_df = data.dropna(subset=[variable]).copy()

    summary = (
        plot_df
        .groupby(["condition", "phase"], observed=True)[variable]
        .agg(mean="mean", std="std", count="count")
        .reset_index()
    )

    summary["sem"] = summary["std"] / np.sqrt(summary["count"])

    x = np.arange(len(PHASE_ORDER))

    for cond in CONDITION_ORDER:
        cond_df = (
            summary[summary["condition"] == cond]
            .set_index("phase")
            .reindex(PHASE_ORDER)
        )

        y = cond_df["mean"].to_numpy(dtype=float)
        e = cond_df["sem"].fillna(0).to_numpy(dtype=float)

        ax.plot(
            x,
            y,
            marker="o",
            linewidth=2,
            color=CONDITION_COLORS[cond],
            label=CONDITION_LABELS[cond],
        )

        ax.fill_between(
            x,
            y - e,
            y + e,
            color=CONDITION_COLORS[cond],
            alpha=0.18,
        )

    ax.axhline(0, linestyle="--", linewidth=1, alpha=0.6)
    ax.set_xticks(x)
    ax.set_xticklabels([PHASE_LABELS[p] for p in PHASE_ORDER], rotation=35, ha="right")
    ax.set_ylabel(ylabel, fontsize=14)
    ax.set_title(title, fontsize=16)
    ax.spines[["top", "right"]].set_visible(False)


# ------------------------------------------------------------
# Figure: dynamic descriptors
# ------------------------------------------------------------

fig, axes = plt.subplots(2, 2, figsize=(20, 13))

# A: intra-phase apparent exchange parameter
add_boxplot_with_points(
    axes[0, 0],
    df,
    variable="k_c_slope_1_per_s",
    ylabel=r"$k_c$ slope [1/s]",
    title=r"(a) Intra-phase apparent exchange parameter $k_c$",
    log_scale=False,
)

# B: cooling rate
add_phase_lineplot(
    axes[0, 1],
    df,
    variable="mean_CR_C_per_s",
    ylabel=r"Mean cooling rate [°C/s]",
    title="(b) Intra-phase cooling rate",
)

# C: apparent cooling index
add_boxplot_with_points(
    axes[1, 0],
    df,
    variable="mean_ACI_1_per_s",
    ylabel=r"Mean ACI [1/s]",
    title="(c) Apparent Cooling Index",
    log_scale=False,
)

# D: cumulative cooling potential / thermal load
add_phase_lineplot(
    axes[1, 1],
    df,
    variable="CCP_deltaT_C_s",
    ylabel=r"CCP$_{\Delta T}$ [°C·s]",
    title=r"(d) Cumulative local thermal gradient",
)

handles, labels = axes[0, 1].get_legend_handles_labels()

fig.legend(
    handles,
    labels,
    loc="upper center",
    ncol=3,
    frameon=False,
    bbox_to_anchor=(0.5, 1.02),
    fontsize=15,
)

fig.suptitle("Dynamic descriptors of intra-phase thermo-hygrometric response", y=1.06)

fig.tight_layout()

out_path = OUTPUT_DIR / "paper_dynamic_descriptors_intra_phase.png"
fig.savefig(out_path, dpi=300, bbox_inches="tight")
plt.close(fig)

print("Saved:", out_path)

/tmp/ipykernel_13026/173622971.py:144: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(
/tmp/ipykernel_13026/173622971.py:144: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(


Saved: /home/eleonorab/repository/convective_cooling/reports/mechanistic_plots/convective_indices_example/paper_dynamic_descriptors_intra_phase.png
